In [ ]:
%pip install scipy numpy matplotlib

# SciPy - Scientific Computing
Statistics, optimization, integration, signal processing, sparse matrices, and linear algebra.

In [ ]:
import scipy
import numpy as np
import matplotlib.pyplot as plt

print('SciPy version:', scipy.__version__)

## 1. Statistical Distributions

In [ ]:
from scipy import stats

# Normal distribution
mu, sigma = 5.0, 2.0
normal = stats.norm(loc=mu, scale=sigma)

x = np.linspace(mu - 4*sigma, mu + 4*sigma, 300)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(x, normal.pdf(x), lw=2, color='steelblue', label='PDF')
axes[0].set_title(f'Normal PDF - mu={mu}, sigma={sigma}')
axes[0].legend()

axes[1].plot(x, normal.cdf(x), lw=2, color='darkorange', label='CDF')
axes[1].set_title('Normal CDF')
axes[1].legend()
plt.tight_layout()
plt.show()

print('Mean:', normal.mean())
print('Variance:', normal.var())
print('P(X < 7):', round(normal.cdf(7), 4))
print('95th percentile:', round(normal.ppf(0.95), 4))

In [ ]:
# Comparing multiple distributions
dists = {
    'Normal':   stats.norm(loc=0, scale=1),
    't (df=5)': stats.t(df=5),
    'Cauchy':   stats.cauchy(),
}

x2 = np.linspace(-5, 5, 300)
fig, ax = plt.subplots(figsize=(8, 4))
for name, dist in dists.items():
    ax.plot(x2, dist.pdf(x2), lw=2, label=name)
ax.set_title('Symmetric Distributions Compared')
ax.legend()
ax.set_ylim(0, 0.45)
plt.tight_layout()
plt.show()

## 2. Hypothesis Testing

In [ ]:
rng = np.random.default_rng(0)
group_a = rng.normal(50, 10, 60)
group_b = rng.normal(54, 10, 60)

# Two-sample t-test (independent)
t_stat, p_val = stats.ttest_ind(group_a, group_b)
print('--- Independent t-test ---')
print(f't-statistic : {t_stat:.4f}')
print(f'p-value     : {p_val:.4f}')
print('Reject H0 (5%):', p_val < 0.05)

# One-sample t-test
t1, p1 = stats.ttest_1samp(group_a, popmean=50)
print('\n--- One-sample t-test (mu=50) ---')
print(f't={t1:.4f}, p={p1:.4f}')

In [ ]:
# Normality tests
sample = rng.normal(0, 1, 200)

stat_sw, p_sw = stats.shapiro(sample[:50])   # Shapiro-Wilk works best <= 50 samples
stat_ks, p_ks = stats.kstest(sample, 'norm', args=(sample.mean(), sample.std()))

print('Shapiro-Wilk  p:', round(p_sw, 4))
print('KS test       p:', round(p_ks, 4))

In [ ]:
# Chi-squared test of independence
observed = np.array([[30, 10], [15, 45]])
chi2, p_chi, dof, expected = stats.chi2_contingency(observed)
print(f'Chi2={chi2:.3f}, p={p_chi:.4f}, dof={dof}')
print('Expected:\n', expected.round(1))

## 3. Optimization

In [ ]:
from scipy.optimize import minimize, minimize_scalar, brentq

# Scalar minimization
f = lambda x: (x - 2.5) ** 2 + 3 * np.sin(x)
result = minimize_scalar(f, bounds=(0, 5), method='bounded')
print('Minimum at x =', round(result.x, 4), '| f(x) =', round(result.fun, 4))

x_plot = np.linspace(0, 5, 300)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(x_plot, f(x_plot), lw=2, color='royalblue')
ax.axvline(result.x, color='crimson', linestyle='--', label=f'min x={result.x:.3f}')
ax.legend()
ax.set_title('Scalar Minimization')
plt.tight_layout()
plt.show()

In [ ]:
# Multivariate - Rosenbrock (banana) function
def rosenbrock(x):
    return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

res = minimize(rosenbrock, x0=[-1.0, 0.5], method='BFGS')
print('Minimum:', res.x.round(6))
print('f(min):', round(res.fun, 8))
print('Converged:', res.success)

In [ ]:
# Root finding
g = lambda x: np.cos(x) - x
root = brentq(g, 0, 2)
print(f'Root of cos(x) - x: x = {root:.8f}')
print('Verification:', round(np.cos(root) - root, 12))

## 4. Numerical Integration

In [ ]:
from scipy.integrate import quad, dblquad

# Single integral
I, err = quad(lambda x: np.exp(-x**2), -np.inf, np.inf)
print('Integral of exp(-x^2) from -inf to inf:', round(I, 8))
print('Expected (sqrt(pi)):', round(np.sqrt(np.pi), 8))

# Double integral
I2, _ = dblquad(lambda y, x: x * y, 0, 1, 0, 1)
print('\nDouble integral of x*y over [0,1]x[0,1]:', I2, '(expected 0.25)')

## 5. Curve Fitting

In [ ]:
from scipy.optimize import curve_fit

def model(x, a, b, c):
    return a * np.exp(-b * x) + c

rng2 = np.random.default_rng(5)
x_data = np.linspace(0, 5, 50)
y_data = model(x_data, a=3.5, b=1.2, c=0.5) + rng2.normal(0, 0.2, 50)

popt, pcov = curve_fit(model, x_data, y_data, p0=[3, 1, 0])
perr = np.sqrt(np.diag(pcov))

print(f'Fitted a={popt[0]:.3f} +/- {perr[0]:.3f}')
print(f'Fitted b={popt[1]:.3f} +/- {perr[1]:.3f}')
print(f'Fitted c={popt[2]:.3f} +/- {perr[2]:.3f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(x_data, y_data, s=15, alpha=0.6, label='Data')
ax.plot(x_data, model(x_data, *popt), color='red', lw=2, label='Fitted curve')
ax.set_title('Curve Fitting - Exponential Decay')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Interpolation

In [ ]:
from scipy.interpolate import interp1d, CubicSpline

x_pts = np.array([0, 1, 2, 3, 4, 5])
y_pts = np.array([0, 1, 4, 2, 3, 5])
x_fine = np.linspace(0, 5, 200)

linear_f = interp1d(x_pts, y_pts, kind='linear')
cubic_cs  = CubicSpline(x_pts, y_pts)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_pts, y_pts, 'o', ms=8, label='Data points')
ax.plot(x_fine, linear_f(x_fine), '--', label='Linear')
ax.plot(x_fine, cubic_cs(x_fine), '-',  label='Cubic Spline', lw=2)
ax.set_title('Interpolation Comparison')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Signal Processing - FFT and Filtering

In [ ]:
from scipy.fft import fft, fftfreq

fs   = 1000          # sampling rate (Hz)
t_s  = np.linspace(0, 1, fs, endpoint=False)
sig  = (np.sin(2 * np.pi * 50 * t_s) +
        0.5 * np.sin(2 * np.pi * 120 * t_s) +
        np.random.default_rng(3).normal(0, 0.3, fs))

freqs = fftfreq(fs, 1/fs)
spectrum = np.abs(fft(sig)) / fs

fig, axes = plt.subplots(2, 1, figsize=(10, 6))
axes[0].plot(t_s[:200], sig[:200], lw=0.8, color='steelblue')
axes[0].set_title('Signal (first 200 samples)')
axes[0].set_xlabel('Time (s)')

pos_mask = freqs > 0
axes[1].plot(freqs[pos_mask], spectrum[pos_mask], lw=1, color='coral')
axes[1].set_title('Frequency Spectrum (FFT)')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_xlim(0, 200)
plt.tight_layout()
plt.show()

## 8. Sparse Matrices

In [ ]:
from scipy.sparse import csr_matrix, eye as speye

dense = np.array([[0, 0, 3, 0],
                  [0, 1, 0, 0],
                  [5, 0, 0, 2],
                  [0, 0, 0, 0]])

sparse = csr_matrix(dense)
print('Dense shape:', dense.shape)
print('Sparse nnz:', sparse.nnz, '(non-zero elements)')
print('Density:', sparse.nnz / (dense.shape[0] * dense.shape[1]))

# Sparse identity
I_sparse = speye(5, format='csr')
print('\nSparse 5x5 identity:')
print(I_sparse.toarray())

## 9. Linear Algebra

In [ ]:
from scipy.linalg import lu, svd, solve

A = np.array([[2., 1., 1.],
              [4., 3., 3.],
              [8., 7., 9.]])
b = np.array([1., 1., 1.])

# Solve linear system
x = solve(A, b)
print('Solution x:', x)
print('Residual:', np.linalg.norm(A @ x - b))

# LU decomposition
P, L, U = lu(A)
print('\nL:\n', L.round(3))
print('U:\n', U.round(3))

# SVD
U_svd, s, Vt = svd(A)
print('\nSingular values:', s.round(4))

## 10. Descriptive Statistics Summary

In [ ]:
sample_data = np.random.default_rng(10).normal(70, 15, 500)

desc = stats.describe(sample_data)
print('n         :', desc.nobs)
print('Min / Max :', round(desc.minmax[0], 2), '/', round(desc.minmax[1], 2))
print('Mean      :', round(desc.mean, 4))
print('Variance  :', round(desc.variance, 4))
print('Skewness  :', round(desc.skewness, 4))
print('Kurtosis  :', round(desc.kurtosis, 4))
print('Median    :', round(np.median(sample_data), 4))
print('IQR       :', round(stats.iqr(sample_data), 4))